In [15]:
import json
import psycopg2
from dateutil import parser as dateparser
import os

In [ ]:
# ---------------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------------
conn = psycopg2.connect(
    host="ep-fancy-dream-a1a6pcdi-pooler.ap-southeast-1.aws.neon.tech",
    database="mates",
    user="neondb_owner",
    password="npg_X0N3vFLwAfEe"
)
conn.autocommit = True
cur = conn.cursor()

In [17]:
# Create normalized database tables
def create_normalized_tables(conn):
    """Create all normalized tables"""
    create_tables_sql = """
    -- Drop tables if they exist (for clean setup)
    DROP TABLE IF EXISTS article_tags CASCADE;
    DROP TABLE IF EXISTS articles CASCADE;
    DROP TABLE IF EXISTS tags CASCADE;
    DROP TABLE IF EXISTS sources CASCADE;
    DROP TABLE IF EXISTS categories CASCADE;

    -- Create categories table
    CREATE TABLE categories (
        category_id SERIAL PRIMARY KEY,
        category_name VARCHAR(100) UNIQUE NOT NULL,
        description TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create sources table
    CREATE TABLE sources (
        source_id SERIAL PRIMARY KEY,
        source_url VARCHAR(500) UNIQUE NOT NULL,
        source_name VARCHAR(200) NOT NULL,
        is_active BOOLEAN DEFAULT TRUE,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create tags table
    CREATE TABLE tags (
        tag_id SERIAL PRIMARY KEY,
        tag_name VARCHAR(100) UNIQUE NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create articles table
    CREATE TABLE articles (
        article_id SERIAL PRIMARY KEY,
        url VARCHAR(1000) UNIQUE NOT NULL,
        source_id INTEGER REFERENCES sources(source_id),
        publication_date DATE NOT NULL,
        scrape_date TIMESTAMP NOT NULL,
        title TEXT NOT NULL,
        content TEXT,
        word_count INTEGER DEFAULT 0,
        sentence_count INTEGER DEFAULT 0,
        character_count INTEGER DEFAULT 0,
        category_id INTEGER REFERENCES categories(category_id),
        category_confidence DECIMAL(3,2) DEFAULT 0.0,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create article_tags junction table
    CREATE TABLE article_tags (
        article_tag_id SERIAL PRIMARY KEY,
        article_id INTEGER REFERENCES articles(article_id) ON DELETE CASCADE,
        tag_id INTEGER REFERENCES tags(tag_id) ON DELETE CASCADE,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE(article_id, tag_id)
    );

    -- Create indexes for better performance
    CREATE INDEX idx_articles_url ON articles(url);
    CREATE INDEX idx_articles_publication_date ON articles(publication_date);
    CREATE INDEX idx_articles_source_id ON articles(source_id);
    CREATE INDEX idx_articles_category_id ON articles(category_id);
    CREATE INDEX idx_articles_created_at ON articles(created_at);
    CREATE INDEX idx_article_tags_article_id ON article_tags(article_id);
    CREATE INDEX idx_article_tags_tag_id ON article_tags(tag_id);
    CREATE INDEX idx_sources_url ON sources(source_url);
    CREATE INDEX idx_categories_name ON categories(category_name);
    CREATE INDEX idx_tags_name ON tags(tag_name);
    """
    
    try:
        cur = conn.cursor()
        cur.execute(create_tables_sql)
        conn.commit()
        cur.close()
        print("Normalized tables created successfully!")
        return True
    except Exception as e:
        print(f"Table creation failed: {e}")
        return False

In [18]:
def get_or_create_source(source_url):
    cur.execute("""
        INSERT INTO sources (source_url, source_name)
        VALUES (%s, %s)
        ON CONFLICT (source_url) DO UPDATE SET source_name = EXCLUDED.source_name
        RETURNING source_id;
    """, (source_url, source_url))
    return cur.fetchone()[0]

In [19]:
def get_or_create_category(category_name):
    if not category_name:
        return None

    cur.execute("""
        INSERT INTO categories (category_name)
        VALUES (%s)
        ON CONFLICT (category_name) DO UPDATE SET category_name = EXCLUDED.category_name
        RETURNING category_id;
    """, (category_name,))
    return cur.fetchone()[0]

In [20]:
def get_or_create_tag(tag_name):
    cur.execute("""
        INSERT INTO tags (tag_name)
        VALUES (%s)
        ON CONFLICT (tag_name) DO UPDATE SET tag_name = EXCLUDED.tag_name
        RETURNING tag_id;
    """, (tag_name,))
    return cur.fetchone()[0]

In [21]:
def insert_article(article_data, source_id, category_id):
    """Insert article into database"""
    import json
    from dateutil import parser as dateparser
    
    publication_date = dateparser.parse(article_data["publication_date"]).date()
    scrape_date = dateparser.parse(article_data["scrape_date"])

    cur.execute("""
        INSERT INTO articles (
            url, source_id, category_id,
            publication_date, scrape_date,
            title, content,
            word_count, sentence_count, character_count,
            category_confidence
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (url) DO UPDATE SET
            title = EXCLUDED.title,
            content = EXCLUDED.content,
            word_count = EXCLUDED.word_count,
            sentence_count = EXCLUDED.sentence_count,
            character_count = EXCLUDED.character_count,
            category_id = EXCLUDED.category_id,
            category_confidence = EXCLUDED.category_confidence
        RETURNING article_id;
    """, (
        article_data["url"],
        source_id,
        category_id,
        publication_date,
        scrape_date,
        article_data["title"],
        article_data["content"],
        article_data.get("word_count", 0),
        article_data.get("sentence_count", 0),
        article_data.get("character_count", 0),
        article_data.get("category_confidence", 0.0)
    ))

    return cur.fetchone()[0]

In [22]:
def insert_article_tags(article_id, tags):
    for tag_name in tags:
        tag_id = get_or_create_tag(tag_name)
        cur.execute("""
            INSERT INTO article_tags (article_id, tag_id)
            VALUES (%s, %s)
            ON CONFLICT (article_id, tag_id) DO NOTHING;
        """, (article_id, tag_id))

In [23]:
# ---------------------------------------------------------
# PROCESS A SINGLE ARTICLE (CORRECT FLOW)
# ---------------------------------------------------------
def process_article(article):
    print(f"→ Processing: {article['title'][:50]}")

    # 1. Insert / get source
    source_id = get_or_create_source(article["source"])

    # 2. Insert / get category
    category_id = get_or_create_category(article.get("primary_category"))

    # 3. Insert article
    article_id = insert_article(article, source_id, category_id)

    # 4. Insert tags
    tags = article.get("tags", [])
    insert_article_tags(article_id, tags)

    print(f"Article inserted with ID {article_id}")

In [24]:
# ---------------------------------------------------------
# MAIN ETL FUNCTION
# ---------------------------------------------------------
def run_etl(json_path):
    print("Loading JSON...")
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Found {len(data)} articles.")

    for article in data:
        try:
            process_article(article)
        except Exception as e:
            print("Error:", e)

    print("ETL Completed Successfully!")

In [25]:
if __name__ == "__main__":
    create_normalized_tables(conn)

    run_etl(r"D:\Menghour\MATES\data\datasets\raw\all_articles_cleaned.json")
    cur.close()
    conn.close()

Normalized tables created successfully!
Loading JSON...
Found 7856 articles.
→ Processing: ចំនួននៃការវាយប្រហារ ទៅលើពលរដ្ឋរុស្ស៊ី នៅក្នុងប្រទេ
Article inserted with ID 1
→ Processing: មន្រ្តីជាន់ខ្ពស់ក្រសួងទេសចរណ៍ ស្នើឱ្យស្ថាប័នពាក់ព័
Article inserted with ID 2
→ Processing: លោក ម៉ឹង យូឡេង ប្រធានមន្ទីរសាធារណការ និងដឹកជញ្ជូនខ
Article inserted with ID 3
→ Processing: កូរ៉េខាងជើង សាកល្បងយានក្រោមទឹក គ្មានមនុស្សបើកបំពាក
Article inserted with ID 4
→ Processing: គេហទំព័រយោធាអាមេរិក ៖ វៀតណាមជាប់ជាប្រទេស ដែលយោធាមា
Article inserted with ID 5
→ Processing: បើកពិធីសម្ពោធពិព័រណ៍ ទំនិញគុណភាព ខ្ពស់អន្តរជាតិ ឆ្
Article inserted with ID 6
→ Processing: មន្រ្តីនាយកដ្ឋាន ដឹកជញ្ជូនរបស់វៀតណាម ដាក់បន្ទុកទៅល
Article inserted with ID 7
→ Processing: ខ្មែរ ប៊ែវើរីជីស លើកកម្ពស់សកម្មភាពការលេងកីឡាដើម្បី
Article inserted with ID 8
→ Processing: យន្តហោះ អ៊ីស្រាអែល វាយប្រហារ ទីតាំងបាញ់រ៉ុកកែត នៅល
Article inserted with ID 9
→ Processing: នាយចឺម ឆាតញ៉ែសង្សារគេ ត្រូវគេលបវាយឡើង បែកក្បាល
Article inserted with ID 10
→ Proc

KeyboardInterrupt: 